# Animationen von Change

Ziel

Für jedes Land und jedes Jahr:

jährliche Änderung in GHG per capita und Vulnerability

Bewegungsvektor und Norm

Flag, ob es eine echte Verbesserung ist (beide Dimensionen ↓)


Output(animationen):
- **Globaler Kontext:** Animation aller Länder über alle Jahre im (GHG per capita, Vulnerability)-Raum zur Einordnung der Gesamtentwicklung.
- **Echte Verbesserungen:** Animation nur jener Jahre, in denen Länder gleichzeitig sinkende GHG-Emissionen pro Kopf und sinkende Vulnerability aufweisen.
- **Stabile Improvers:** Fokussierte Animation der Länder mit nachhaltiger und substanzieller Verbesserung über mehrere Jahre hinweg.



In [18]:
import numpy as np
import pandas as pd





## Movement DataFrame: jährliche Dynamik pro Land

**Ziel**
Erzeuge aus dem Rohdatensatz eine **jährliche Bewegungsbeschreibung** pro Land, um Veränderungen in Emissionen und Vulnerability explizit analysieren zu können.

**Schritte**
1. Sortiere die Daten nach `ISO3` und `Year`, um die zeitliche Reihenfolge pro Land sicherzustellen.
2. Berechne jährliche Veränderungen pro Land:
   - `d_ghg`: Änderung der Emissionen pro Kopf gegenüber dem Vorjahr
   - `d_vuln`: Änderung der Vulnerability gegenüber dem Vorjahr
3. Interpretiere diese Änderungen als Bewegungsvektor im 2D-Raum.
4. Berechne die Bewegungsstärke als euklidische Norm:
   \[
   \sqrt{d\_ghg^2 + d\_vuln^2}
   \]
5. Markiere **echte Verbesserungen**:
   - `both_improved = True`, wenn beide Dimensionen abnehmen.

**Logik**


In [8]:



def build_movement_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Sortieren ist kritisch für korrekte Deltas
    df = df.sort_values(["ISO3", "Year"])

    # Jährliche Deltas pro Land
    df["d_ghg"] = df.groupby("ISO3")["GHG_per_capita"].diff()
    df["d_vuln"] = df.groupby("ISO3")["Vulnerability"].diff()

    # Bewegungsnorm
    df["movement_norm"] = np.sqrt(df["d_ghg"]**2 + df["d_vuln"]**2)

    # Strenge Verbesserung (beide Dimensionen besser)
    df["both_improved"] = (df["d_ghg"] < 0) & (df["d_vuln"] < 0)

    # Richtung explizit (optional, hilfreich später)
    df["dir_left"] = df["d_ghg"] < 0
    df["dir_down"] = df["d_vuln"] < 0

    return df




In [14]:

merged_df = pd.read_csv("data/merged_df.csv")
movement_df = build_movement_df(merged_df)
movement_df.head(10)

,ISO3,Country,Year,GHG_per_capita,ND_GAIN,Vulnerability,GDP_per_capita,d_ghg,d_vuln,movement_norm,both_improved,dir_left,dir_down
0,AFG,Afghanistan,1995,0.728879,34.783530,0.612511,813.550256,NaN,NaN,NaN,False,False,False
1,AFG,Afghanistan,1996,0.770651,34.775074,0.612679,813.550256,0.041772,0.000168,0.041773,False,False,False
2,AFG,Afghanistan,1997,0.805011,34.988812,0.612837,813.550256,0.034360,0.000158,0.034361,False,False,False
3,AFG,Afghanistan,1998,0.829508,35.293407,0.611179,813.550256,0.024496,-0.001659,0.024552,False,False,True
4,AFG,Afghanistan,1999,0.850790,35.177507,0.609828,813.550256,0.021282,-0.001351,0.021325,False,False,True
5,AFG,Afghanistan,2000,0.714894,35.065559,0.608398,813.550256,-0.135896,-0.001430,0.135904,True,True,True
6,AFG,Afghanistan,2001,0.602315,35.198269,0.606533,747.688045,-0.112579,-0.001865,0.112594,True,True,True
7,AFG,Afghanistan,2002,0.692841,35.335123,0.604601,926.507941,0.090525,-0.001931,0.090546,False,False,True
8,AFG,Afghanistan,2003,0.687519,35.542042,0.593451,966.962032,-0.005321,-0.011150,0.012355,True,True,True
9,AFG,Afghanistan,2004,0.663101,35.409770,0.595817,971.633503,-0.024418,0.002366,0.024533,False,True,False


## Ranking: Länder nach nachhaltiger Verbesserung

**Ziel**
Aggregiere die jährlichen Bewegungen zu **Länderkennzahlen**, um Länder nach Dauer und Stärke ihrer normativen Verbesserung zu vergleichen.

**Schritte**
1. Beschränke die Daten auf Jahre mit **echter Verbesserung**
   (`both_improved == True`).

2. Aggregiere pro Land:
   - `improvement_years`: Anzahl Jahre mit gleichzeitiger Verbesserung
   - `cumulative_movement`: Summe der Bewegungsstärken über diese Jahre
   - `avg_movement`: durchschnittliche jährliche Bewegungsstärke
   - `first_year`, `last_year`: Zeitraum der Verbesserungsphase

3. Sortiere Länder absteigend nach `cumulative_movement`.

**Logik**
- Es werden nur Jahre berücksichtigt, in denen sich ein Land **in beiden Dimensionen verbessert**.
- Die Kombination aus Anzahl der Jahre und Bewegungsstärke erfasst **nachhaltige statt kurzfristige** Verbesserungen.
- Die Sortierung priorisiert Länder mit der **grössten kumulativen Bewegung Richtung links unten**.
---

### Ranking: Länder nach nachhaltiger Verbesserung (mathematische Definition)

Sei für Land $i$ und Jahr $t$:

- $G_{i,t}$: GHG per capita
- $V_{i,t}$: Vulnerability

**Jährliche Änderungen**
$$
\Delta G_{i,t} = G_{i,t} - G_{i,t-1}
$$
$$
\Delta V_{i,t} = V_{i,t} - V_{i,t-1}
$$

**Strenge Verbesserung**
$$
I_{i,t} =
\begin{cases}
1, & \text{falls } \Delta G_{i,t} < 0 \;\land\; \Delta V_{i,t} < 0 \\
0, & \text{sonst}
\end{cases}
$$

**Bewegungsstärke (Norm)**
$$
\lVert \mathbf{d}_{i,t} \rVert
=
\sqrt{(\Delta G_{i,t})^2 + (\Delta V_{i,t})^2}
$$

**Anzahl Verbesserungsjahre**
$$
N_i = \sum_t I_{i,t}
$$

**Kumulative Verbesserung**
$$
S_i = \sum_{t:\, I_{i,t}=1} \lVert \mathbf{d}_{i,t} \rVert
$$

**Durchschnittliche jährliche Verbesserung**
$$
\bar{S}_i = \frac{S_i}{N_i}
$$

Diese Grössen bilden die Grundlage für das Länder-Ranking nach **Dauer** und **Stärke** der nachhaltigen Verbesserung.



In [12]:
def build_country_improvement_ranking(movement_df: pd.DataFrame) -> pd.DataFrame:
    df = movement_df.copy()

    # Nur echte Verbesserungsjahre zählen
    improved = df[df["both_improved"]].copy()

    ranking = (
        improved
        .groupby(["ISO3", "Country"])
        .agg(
            improvement_years=("Year", "count"),
            cumulative_movement=("movement_norm", "sum"),
            avg_movement=("movement_norm", "mean"),
            first_year=("Year", "min"),
            last_year=("Year", "max"),
        )
        .reset_index()
        .sort_values("cumulative_movement", ascending=False)
    )

    return ranking


In [13]:
country_ranking = build_country_improvement_ranking(movement_df)
country_ranking.head(10)


,ISO3,Country,improvement_years,cumulative_movement,avg_movement,first_year,last_year
130,PLW,Palau,9,227.130517,25.236724,1996,2016
136,QAT,Qatar,15,57.587028,3.839135,1998,2022
16,BHR,Bahrain,12,27.651835,2.304320,1997,2023
91,KWT,Kuwait,10,19.221255,1.922126,1998,2019
65,GNQ,Equatorial Guinea,13,17.373739,1.336441,2002,2023
24,BRN,Brunei,9,15.126264,1.680696,2002,2021
58,GAB,Gabon,11,13.699311,1.245392,1997,2021
53,EST,Estonia,6,11.949249,1.991542,1999,2023
100,LUX,Luxembourg,10,11.337282,1.133728,1996,2016
46,DNK,Denmark,14,11.166211,0.797586,1997,2023


## Filter: stabile Improvers (Länderauswahl)

**Ziel**
Identifiziere Länder mit nachhaltiger und substanzieller Verbesserung.

**Schritte**
1. Filtere Länder mit mindestens 5 Verbesserungsjahren:
   `improvement_years >= 5`

2. Filtere zusätzlich nach Stärke der Bewegung:
   `avg_movement > median(avg_movement)`

**Logik**
- Schritt 1 schliesst Zufall und Einjahres-Effekte aus.
- Schritt 2 schliesst triviale, kaum sichtbare Bewegungen aus.

→ Übrig bleiben Länder mit dauerhafter und relevanter Entwicklung.


In [15]:
stable_improvers = country_ranking[
    (country_ranking["improvement_years"] >= 5) &
    (country_ranking["avg_movement"] > country_ranking["avg_movement"].median())
]


## Trajektorien-DataFrame für Animation

**Ziel**
Erzeuge ein DataFrame mit **jährlichen Trajektorien** für genau jene Länder, die als *stabile Improvers* identifiziert wurden.

**Schritte**
1. Extrahiere die ISO-Codes der ausgewählten Länder aus `stable_improvers`.
2. Filtere `movement_df` auf diese Länder.
3. Sortiere nach `ISO3` und `Year`, um eine korrekte zeitliche Abfolge sicherzustellen.

**Logik**
- Das Ranking erfolgt auf **Länder-Ebene**, die Animation benötigt **Zeit-Ebene**.
- Dieser Schritt verbindet beides: Er überführt die Länderauswahl in **Jahr-für-Jahr-Trajektorien**.
- Das resultierende DataFrame bildet die direkte Grundlage für alle folgenden Animationen.


In [16]:
def build_trajectories_df(
    movement_df: pd.DataFrame,
    stable_improvers: pd.DataFrame
) -> pd.DataFrame:
    iso_keep = stable_improvers["ISO3"].unique()
    traj_df = movement_df[movement_df["ISO3"].isin(iso_keep)].copy()
    traj_df = traj_df.sort_values(["ISO3", "Year"])
    return traj_df


trajectories_df = build_trajectories_df(movement_df, stable_improvers)



## Fixe Achsen für die Animation festlegen

**Ziel**
Definiere ein **stabiles Koordinatensystem** für alle Frames der Animation, damit Bewegungen über die Zeit **vergleichbar** und **nicht verzerrt** dargestellt werden.

**Schritte**
1. Bestimme robuste Achsgrenzen für:
   - x-Achse: `GHG_per_capita`
   - y-Achse: `Vulnerability`
2. Verwende Quantile (z. B. 2 %–98 %), um Ausreisser zu dämpfen.
3. Nutze diese Grenzen **für alle Frames identisch**.

**Logik**
- Variable Achsen erzeugen visuelle Artefakte (scheinbare Sprünge).
- Fixe Achsen machen Trajektorien **lesbar, ruhig und ehrlich**.
- Quantile schützen vor Extremwerten, ohne die Struktur zu verlieren.
